Script: this piece of code loads and processes precipitation files for single inputs for use later in plotting scripts

Notes: needs to be run individually for each model - check the boxes that have a *change me* tag on the top. 

For each model the tas and pr fields with the ensemble mean removed. 
DJFerem




In [1]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from natsort import natsorted 

In [ ]:
#set up the data directory and load in lon + lat + time dimensions
outputdirb='/glade/campaign/cgd/cas/nmaher/cesm2_lens/Amon/pr/' 

outputdir2='/glade/work/nmaher/SEOF_output/'

model='CESM2'

#option for single files - put one file path here to get lon/lat/time
ds_fx = xr.open_dataset(outputdirb+'pr_mon_CESM2_cmip6_hist_ssp370_r1001.001i1p1f1_g025.nc')

lon = ds_fx.lon
lat = ds_fx.lat
time = ds_fx.time

In [ ]:
#*change me*

#set up list of files for each member
filesb = natsorted(os.listdir(outputdirb))
filesb = [s for s in filesb if "_g025" in s]


n = len(filesb)
ne = np.empty(n)
for ii in range(n):
        ne[ii] = ii
      

In [ ]:
#loop through ensemble members and load data
pr_all = np.empty((n,len(time),len(lat),len(lon)))
pr_all = xr.DataArray(pr_all, coords=[ne, time, lat, lon], dims=["member", "time", "lat", "lon"])

for ii in range(n):
        filenameH = outputdirb+filesb[ii]
        ds_member = xr.open_dataset(filenameH)
        pr = ds_member.pr
        pr_all[ii,:,:,:] = np.squeeze(pr.values)


In [ ]:
#select season
pr_DJF_full = pr_all.where(pr_all['time.season'] == 'DJF')

In [ ]:
#*change me*

#piece might be needed for CESM2?
pr_DJF_full*1000

In [ ]:
#take seasonal mean for masked and full regions
pr_DJF_full = pr_DJF_full.rolling(min_periods=3, center=True, time=3).mean()

#make annual mean
pr_DJF_full = pr_DJF_full.groupby('time.year').mean('time')


In [ ]:
#set up the input and the full - remove first timestep as not DJF but JF and compute the anomalies around the ensemble mean
pr_DJF_2_full=pr_DJF_full[:,1:,:,:] 
pr_DJF_2_full=pr_DJF_2_full.values
pr_DJF_2e_full = pr_DJF_2_full - np.ma.average(pr_DJF_2_full,axis=0)[np.newaxis,:,:,:]


In [ ]:
#save DJF emeanremoved zg
eof_T='_prDJFerem'
np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=pr_DJF_2e_full.data, mask=pr_DJF_2e_full.mask)
